# 🎮 2D游戏素材生成工具 - Google Colab部署

## 特点
- ✅ 免费T4 GPU (生成速度: 2-3秒/张)
- ✅ 包含七牛云风格缓存功能
- ✅ 支持Gradio公网访问

## 使用说明
1. 点击菜单: 运行时 → 更改运行时类型 → 选择 GPU (T4)
2. 按顺序运行下面的代码块
3. 等待最后一个单元格输出Gradio链接
4. 点击链接即可使用

In [ ]:
# 步骤1: 检查并克隆项目代码
import os

# 检查项目是否已存在
if os.path.exists('2d-game-assets-generator'):
    print("✅ 项目已存在,跳过克隆")
    %cd 2d-game-assets-generator
else:
    print("📥 克隆项目...")
    !git clone https://github.com/lsthaha/2d-game-assets-generator.git
    %cd 2d-game-assets-generator
    print("✅ 克隆完成")

# 验证目录结构
!ls -la
!ls -la backend/ frontend/

In [ ]:
# 步骤2: 安装依赖 (约3-5分钟)
!pip install -q torch torchvision diffusers transformers gradio fastapi uvicorn \
    pillow opencv-python pyyaml python-dotenv peft safetensors requests pydantic qiniu tqdm

In [ ]:
# 步骤3: 配置为GPU模式
config_content = '''
# GPU配置
MODEL_CONFIG = {
    "base_model": "runwayml/stable-diffusion-v1-5",
    "lcm_lora": "latent-consistency/lcm-lora-sdv1-5",
    "device": "cuda",  # Colab GPU
    "dtype": "float16",  # GPU加速
    "enable_attention_slicing": True,
    "enable_memory_efficient_attention": True,
    "background_removal_method": "rembg",
}
'''

# 修改config.py
!sed -i 's/"device": "cpu"/"device": "cuda"/' config.py
!sed -i 's/"dtype": "float32"/"dtype": "float16"/' config.py
!sed -i 's/"enable_memory_efficient_attention": False/"enable_memory_efficient_attention": True/' config.py

print("✅ GPU配置完成")

In [ ]:
# 步骤4: 启动后端 (后台运行)
import subprocess
import time

# 后台启动后端
backend_process = subprocess.Popen(
    ['python', 'backend/main.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

print("⏳ 启动后端服务...")
time.sleep(15)  # 等待后端启动
print("✅ 后端已启动在 http://127.0.0.1:8001")

In [ ]:
# 步骤5: 修改并启动前端 (自动生成公网链接)
import os

# 检查文件是否存在
if not os.path.exists('frontend/app.py'):
    print("❌ 错误: frontend/app.py 不存在!")
    print("当前目录:", os.getcwd())
    !ls -la
else:
    print("✅ 找到 frontend/app.py")
    
    # 修改为公网模式
    with open('frontend/app.py', 'r', encoding='utf-8') as f:
        content = f.read()
    
    # 修改launch参数 (将share=False改为share=True)
    if 'share=False' in content:
        content = content.replace('share=False', 'share=True')
        
        with open('frontend/app.py', 'w', encoding='utf-8') as f:
            f.write(content)
        print("✅ 已启用公网分享模式")
    elif 'share=True' in content:
        print("✅ 已经是公网模式")
    else:
        print("⚠️  未找到share参数,使用默认配置")
    
    # 启动前端
    print("\n🚀 启动前端...")
    print("⏳ 等待生成公网链接 (约10-20秒)")
    print("📌 链接生成后,点击即可访问!\n")
    !python frontend/app.py

## 🎯 使用提示

### 性能对比
- **GPU (T4)**: 2-3秒/张 ⚡
- **CPU**: 2-4分钟/张 🐢

### 推荐配置
- 推理步数: 4-8步
- 引导强度: 7.5
- 使用种子锁定保持风格一致

### 七牛云功能
如需启用七牛云集成:
1. 在Colab中设置环境变量
2. 生成的素材会自动上传到Kodo
3. 风格缓存会同步到云端

```python
import os
os.environ['QINIU_ACCESS_KEY'] = 'your_key'
os.environ['QINIU_SECRET_KEY'] = 'your_secret'
os.environ['QINIU_BUCKET'] = 'your_bucket'
```

## 🔧 完整重启方案 (如果遇到问题)

如果前面的步骤出现问题,按顺序运行以下代码块:

In [ ]:
# 重启步骤1: 清理所有进程
!pkill -f "main.py"
!pkill -f "app.py"
import time
time.sleep(3)
print("✅ 所有进程已清理")

In [ ]:
# 重启步骤2: 确认目录和配置
import os
os.chdir('/content/2d-game-assets-generator')
print(f"📂 当前目录: {os.getcwd()}")
print("\n📋 文件列表:")
!ls -la backend/
print("\n⚙️ GPU配置:")
!grep "device" config.py | head -3

In [ ]:
# 重启步骤3: 检查CUDA
import torch
print(f"🎯 PyTorch版本: {torch.__version__}")
print(f"🎯 CUDA可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎯 GPU型号: {torch.cuda.get_device_name(0)}")
    print(f"🎯 显存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("⚠️ 警告: CUDA不可用,请检查运行时类型!")

In [ ]:
# 重启步骤4: 启动后端(带详细日志)
import subprocess
import time

print("⏳ 启动后端服务...")
print("-" * 60)

# 启动后端并显示输出
process = subprocess.Popen(
    ['python', '-u', 'backend/main.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    universal_newlines=True,
    bufsize=1
)

# 显示启动日志(前40行)
for i in range(40):
    line = process.stdout.readline()
    if line:
        print(line.strip())
        # 检测到成功启动的关键词
        if "Uvicorn running" in line or "Application startup complete" in line:
            print("\n" + "=" * 60)
            print("✅ 后端启动成功!")
            print("=" * 60)
            break
    time.sleep(0.5)

print("\n⏳ 等待后端完全就绪...")
time.sleep(5)

In [ ]:
# 重启步骤5: 测试后端连接
import requests
import time

print("🔍 测试后端连接...\n")

for i in range(5):
    try:
        response = requests.get("http://127.0.0.1:8001/health", timeout=5)
        if response.status_code == 200:
            print(f"✅ 后端连接成功!")
            print(f"📊 响应: {response.json()}")
            print(f"\n🎉 后端API地址: http://127.0.0.1:8001")
            print(f"📚 API文档: http://127.0.0.1:8001/docs")
            break
    except Exception as e:
        print(f"⏳ 尝试 {i+1}/5 - 等待后端启动...")
        if i == 4:
            print(f"\n❌ 后端未能启动")
            print(f"错误信息: {e}")
            print("\n💡 建议:")
            print("1. 检查是否选择了GPU运行时")
            print("2. 重新运行前面的安装步骤")
            print("3. 查看上面的日志输出查找错误")
        time.sleep(5)

In [ ]:
# 重启步骤6: 快速测试生成功能
import requests
from PIL import Image
import io

print("🎨 测试素材生成功能...\n")

# 发送生成请求
response = requests.post(
    "http://127.0.0.1:8001/generate",
    json={
        "prompt": "a red knight warrior, pixel art style, high quality",
        "asset_type": "character_portrait",
        "style": "pixel_art",
        "width": 256,
        "height": 256,
        "num_inference_steps": 4,
        "guidance_scale": 7.5,
        "remove_background": False,
        "seed": 42
    },
    timeout=120
)

if response.status_code == 200:
    result = response.json()
    print("✅ 生成成功!")
    print(f"📌 任务ID: {result.get('task_id')}")
    print(f"🎲 种子值: {result.get('seed')}")
    print(f"⏱️  生成时间: {result.get('generation_time', 'N/A')}秒")
    
    # 显示图片
    if result.get('image_paths'):
        img = Image.open(result['image_paths'][0])
        print(f"🖼️  图片尺寸: {img.size}")
        display(img)
        print("\n🎉 后端功能正常! 现在可以启动前端了")
else:
    print(f"❌ 生成失败: {response.status_code}")
    print(f"错误信息: {response.text}")

In [ ]:
# 重启步骤7: 启动前端(公网模式)
print("🌐 启动前端界面(公网访问模式)...\n")

# 修改app.py启用公网分享
import fileinput
import sys

with open('frontend/app.py', 'r', encoding='utf-8') as f:
    content = f.read()

# 确保启用share=True
if 'share=True' not in content:
    content = content.replace(
        'demo.launch()',
        'demo.launch(share=True, server_port=7860)'
    )
    with open('frontend/app.py', 'w', encoding='utf-8') as f:
        f.write(content)
    print("✅ 已启用公网访问模式")

print("\n⏳ 启动前端服务...")
print("📝 提示: 此单元格会持续运行,显示Gradio链接")
print("🔗 请在输出中找到 'Running on public URL' 并点击链接\n")
print("=" * 60)

# 启动前端
!python frontend/app.py

## 📊 风格缓存功能测试

测试七牛云风格缓存命中率:

In [ ]:
# 测试风格缓存
from backend.integrations import StyleCacheManager, generate_style_cache_key
from pathlib import Path

# 创建缓存管理器
manager = StyleCacheManager(cache_dir=Path("cache"))

# 测试配置
style_config = {
    "美术风格": "pixel_art",
    "色彩主调": "warm_tones",
    "固定种子": 42,
    "像素尺寸": "16px",
    "线条粗细": "medium",
    "饱和度": 75,
    "素材类型": "character"
}

# 生成缓存键
cache_key = generate_style_cache_key(style_config)
print(f"缓存键: {cache_key}")

# 预测命中率
hit_rate, suggestion = manager.calculate_hit_rate_prediction(style_config)
print(f"\n预测命中率: {hit_rate*100:.1f}%")
print(f"建议: {suggestion}")

# 检查缓存
is_hit, level, data = manager.check_cache(style_config)
if is_hit:
    print(f"\n✅ 缓存命中! 级别: {level}")
else:
    print(f"\n❌ 缓存未命中,需要生成新风格")